# 结构化输出

| 维度 | 模型的结构化输出 | Agent 结构化输出 |
| ---- | ---- | ---- |
| 操作对象 | 作用于**大模型对象** | 作用于**Agent** |
| 解析时机 | 每次模型调用生成 AIMessage 时，进行解析 | 仅在 Agent 决定“任务结束”并输出最终答案时解析 |
| 数据流转 | 模型 → 结构化对象 | 模型 → 工具 → 反思 → … → 结构化对象 |
| 绑定方式 | 使用 `with_structured_output` | 使用 `response_format` 参数 |
| 适用场景 | **单次、确定性**的任务（如提取字段、翻译、分类） | **多步、复杂推理**的任务（如查文档后汇总报表） |

##  ProviderStrategy
使用模型提供商的**原生结构化输出功能**实现结构化输出。(NLP课题：LLM可控生成)
- 这里所说的“原生结构化输出”指的是大语言模型（LLM）提供商通过其API直接提供的、在模型响应阶段就强制保证**输出格式符合预定规范**的能力，这种能力能够在模型生成内容的源头确保结构化准确性。
- 适用于支持原生结构化输出的模型，比如OpenAI、Anthropic Claude或xAI Grok等。


运行下面代码可以看见 OpenAI模型ProviderStrategy调用后，直接返回结构化输出。

DeepSeek、ZAI等模型，ProviderStrategy会直接报错，需要关闭思考模型，同时使用ToolStrategy。

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

# 2.Pydantic结构化方式定义
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3.agent初始化
agent = create_agent(
    model=model,
    response_format=ProviderStrategy(ContactInfo)
)

# 4.调用
response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
})

rprint(response)


{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='d90ba01d-ba4c-4e7f-add5-4a99d1d4cfaf'
        ),
        AIMessage(
            content='{"name":"小明","email":"shkstart@atguigu.com","phone":"12345678912"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 31,
                    'prompt_tokens': 154,
                    'total_tokens': 185,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.000255,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.000255,
                        'upstream_inference_prompt_cost': 0.0001155,
                        'upstream_inference_completions_cost': 0.0001395
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784217189-Uytzq2ZAtNHAi1stROMf',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f6ba1-a0af-7bf2-a87d-b31a9a7447af-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 154,
                'output_tokens': 31,
                'total_tokens': 185,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

In [5]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

model_ds = init_chat_model(
    model = "deepseek-v4-pro",
    extra_body={"thinking": {"type": "disabled"}}
)



# 3.agent初始化
agent_ds = create_agent(
    model=model_ds,
    response_format=ToolStrategy(ContactInfo),
)

# 4.调用
response_ds = agent_ds.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
})

rprint(response_ds)


{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='6e61da96-7b40-4713-8100-48fd9af128f0'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 351,
                    'total_tokens': 429,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 351
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-pro',
                'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
                'id': 'a3bd0db6-7489-4fef-86f5-c7c2d2572aec',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6ba8-3ade-7ee2-b757-6193a24d9137-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_00_1lSM5MlmcQMbkPjlz2Jw0862',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 351,
                'output_tokens': 78,
                'total_tokens': 429,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='8b09e037-c171-489c-a9c9-6e9818108a0f',
            tool_call_id='call_00_1lSM5MlmcQMbkPjlz2Jw0862'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

In [13]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

model_zai = ChatOpenAI(
    temperature=0.6,
    model="glm-5.2",
    openai_api_key=os.getenv("ZAI_API_KEY"),
    openai_api_base=os.getenv("ZAI_BASE_URL")
)


agent_zai = create_agent(
    model=model_zai,
    response_format=ToolStrategy(ContactInfo)
)

# 4.调用
response_zai = agent_zai.invoke({   
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
})

rprint(response_zai)



{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='c0e6ae2b-aba8-4c16-bbc6-925a019f14af'
        ),
        AIMessage(
            content='我来从这段话中提取结构化信息并保存：\n\n- **姓名**：小明\n- 
**邮箱地址**：shkstart@atguigu.com\n- **手机号**：12345678912',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 141,
                    'prompt_tokens': 230,
                    'total_tokens': 371,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 59,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.2',
                'system_fingerprint': None,
                'id': '202607170018487848dc93064045fa',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6bb9-31e0-78d3-8ec2-604976738cfa-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_-7438254779618291706',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 230,
                'output_tokens': 141,
                'total_tokens': 371,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 59}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='cb8ac4dc-03a7-4347-affc-e99f9038700d',
            tool_call_id='call_-7438254779618291706'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

## ToolStrategy

工具策略适应性更广泛，详细解析此方法

ToolStrategy通过**工具调用**（Tool Calling/Function Calling）实现结构化输出，所以LangChain会在消息列表末尾追加一条`ToolMessage`，让整个链路完整。但实际上没有实际的工具执行，这是一条伪消息。

ToolStrategy适用于任何支持工具调用的现代模型。

ToolStrategy的配置包含三个主要参数：
```python
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[bool, str, type[Exception], tuple[type[Exception], ...], Callable[[Exception], str]]
```

- **schema（必需参数）**：与提供商策略的schema参数功能一致，支持Pydantic模型、TypedDict、JSON Schema、数据类(`@dataclass`)，同时还支持联合类型 `Union[类型1, 类型2]`（允许模型根据输入内容选择最匹配的数据结构）。
- **tool_message_content（可选参数）**：用于自定义生成结构化输出时，会话历史中记录的提示信息。默认使用展示输出数据的标准响应语句。
- **handle_errors（可选参数）**：用于指定数据校验失败时的重试策略，默认值为`True`。



### 四个Schema模式

与模型结构化输出相同

**输出模式1：Pydantic类型**

Pydantic类型的Schema支持数据验证，是优先推荐使用的方式。

**输出模式2：TypedDict类型**

TypedDict允许为字典对象定义固定的键名和对应的值类型，是带有类型提示的字典结构。具体规则：
1. TypedDict字段定义采用 `Annotated[类型, 默认值, "描述"]` 格式
2. 可选字段使用 `Optional` 包装，默认值在Annotated中指定
3. TypedDict **不支持运行时验证**

**输出模式3：JsonSchema类型**

JSON Schema通过传入标准JSON Schema字典定义数据结构，适合多编程语言交互、需要编写复杂数据约束的场景。

**输出模式4：@dataclass类型**

`@dataclass` 是Python3.7新增的装饰器，作用是简化数据存储类的定义。


In [2]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_BASE_URL")

model = init_chat_model(
    model="openai:gpt-5.4-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录: 张三，VIP客户，最近购买日期: 2026-01-15，累计消费: $15,000"
    elif "李四" in query.lower():
        return "客户记录: 李四，普通客户，最近购买日期: 2025-12-20，累计消费: $3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] = Field("潜在客户", description="客户等级，只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="""
请分析指定客户的情况：
1. 先搜索客户数据库了解最新情况
2. 如果是VIP客户，则发送感谢邮件
3. 基于搜索结果生成结构化分析报告
4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件
"""),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)


if "structured_response" in result:
    analysis = result["structured_response"]
    print(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='905cec81-bf7c-4f6e-8b76-20fc6a2b8026'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 20,
                    'prompt_tokens': 289,
                    'total_tokens': 309,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00030675,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00030675,
                        'upstream_inference_prompt_cost': 0.00021675,
                        'upstream_inference_completions_cost': 9e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784278542-uTJhhfAin0ZFq41G1sy8',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f6f49-d28e-72f2-a879-4c2b37157b12-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_tPzGxRAIN6ID1zliSBU6Gv7N',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 289,
                'output_tokens': 20,
                'total_tokens': 309,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='客户记录: 张三，VIP客户，最近购买日期: 2026-01-15，累计消费: $15,000',
            name='search_customer_database',
            id='ef8e249e-4a68-4335-9d4d-b158821f6d6c',
            tool_call_id='call_tPzGxRAIN6ID1zliSBU6Gv7N'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 19,
                    'prompt_tokens': 349,
                    'total_tokens': 368,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00034725,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00034725,
                        'upstream_inference_prompt_cost': 0.00026175,
                        'upstream_inference_completions_cost': 8.55e-05
                    }
                },
                'model_provider': 'openai',
                'model_name': '

customer_name='张三' customer_tier='VIP客户' recent_activity='最近购买日期: 2026-01-15' spending_level='高' send_email=True


### 多schema联合模式

ToolStrategy允许指定多个类型`Union[类型1,类型2]`这种写法，LLM能够根据输入文本的内容，智能地选择**最合适的一个**数据模型（Schema）来生成结构化输出，但是最终只会有一种类型输出。
适用于根据不同输入内容，生成不同的结构化输出的场景，但是底层工具转换结构化输出只会转换成一种结构化类型输出。

1. 定义两个Pydantic结构体：`ContactInfo`（联系人信息）、`EventInfo`（事件信息）
2. 通过`ToolStrategy + Union`实现多Schema联合，大模型会自动判断输入文本匹配哪一个结构体，**只输出其中一种结构**
3. 示例输入为小明邮箱+手机号，会自动解析并返回`ContactInfo`类型的结构化结果


In [2]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_BASE_URL")

model = init_chat_model(
    model="openai:gpt-5.4-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventInfo]
    )
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
        ]
    }
)

rprint(response)


{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='ac2a9b2e-4509-406b-abba-146dd22f9a4a'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 36,
                    'prompt_tokens': 125,
                    'total_tokens': 161,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00025575,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00025575,
                        'upstream_inference_prompt_cost': 9.375e-05,
                        'upstream_inference_completions_cost': 0.000162
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784303748-HDobdQFxItFLylvJSm5s',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f70ca-801e-7611-a8aa-9b1dc22c51e8-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_3hzuwAyle6rX8ToYkFitHbHP',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 125,
                'output_tokens': 36,
                'total_tokens': 161,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='6d4e6cad-2247-449d-9a24-09533288a89e',
            tool_call_id='call_3hzuwAyle6rX8ToYkFitHbHP'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

### 自定义工具消息

在ToolStrategy中，通常模型利用Function Call调用工具进行结构化输出之后，工具会返回一个工具消息，通常默认是结构化的JSON

但是当Schema特别复杂时，工具返回的内容会非常多，但这个信息实际上没有什么用处，可以自定义工具消息内容。

通过ToolStrategy的 **tool_message_content** 参数定制其消息内容，将指定的内容写入对话历史的提示信息，这样做的好处如下：
1. 在最终用户可见的对话流中，使用**更自然的消息**替代原始数据。
2. 用简短的确认信息替代可能很长的数据块，**减少token消耗**。

当不设置 tool_message_content时，模型收到的 ToolMessage里就包含了像 `{'name': '张三', 'email': 'zhangsan@email.com'...}` 这样的具体数据。当设置了tool_message_content时，模型收到的ToolMessage只是一个预定义的确认信息，如**“格式化输出成功！”**。这种方式节省了上下文窗口的令牌消耗，并且让对话流对最终用户更友好。

说明：无论 tool_message_content如何设置，成功提取的结构化数据最终都会正确存入
`result["structured_response"]` 返回，自定义消息仅影响对话历史中的一条记录。

In [3]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from rich import print as rprint


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo, tool_message_content="格式化输出成功！")
)

response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='2a890fb0-b30a-40b8-b5c0-77b68dbf199c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 35,
                    'prompt_tokens': 91,
                    'total_tokens': 126,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00022575,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00022575,
                        'upstream_inference_prompt_cost': 6.825e-05,
                        'upstream_inference_completions_cost': 0.0001575
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784304216-VoMd3mnLyk4TCWfUmoGd',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f70d1-a76a-7690-b164-23617ea60acb-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_TqAHURUmkHJ1g9G79FYbNTie',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 91,
                'output_tokens': 35,
                'total_tokens': 126,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='格式化输出成功！',
            name='ContactInfo',
            id='09b3eb13-8239-4eab-80fc-570061a76e80',
            tool_call_id='call_TqAHURUmkHJ1g9G79FYbNTie'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

### 错误处理：handle_errors参数

受限于模型能力，大模型输出的内容可能并不符合格式要求，ToolStrategy通过其handle_errors参数提供了结构化过程错误处理策略，以下是主要的几种方式及其用途：

- handle_errors=True：LangChain默认方式，捕获所有异常，并使用LangChain内置的、信息明确的错误消息模板提示模型重试，确保最终能得到符合预定格式的有效数据。适用于大多数希望自动处理错误的通用场景。
- handle_errors=False：关闭自动重试机制，任何异常都会直接抛出，会中断程序运行。
- handle_errors="自定义字符串"：捕获所有异常，但使用开发者预设的固定字符串作为错误消息。适用于需要统一、友好的用户提示，或进行特定业务引导的场景。
- handle_errors=ExceptionType：仅捕获指定类型(如ValueError)或元组中的异常类型并进行重试，其他异常直接抛出。适用于需要精准控制，只对特定错误进行重试的场景。
- handle_errors=callable：灵活性最高的方式，使用开发者自定义的函数来处理异常，可根据不同的异常类型返回差异化的提示信息。适用于需要复杂、精细化错误处理的场景。

#### 情况1：设置为True/False/固定字符串 

构造报错：模型对于单条信息的格式化输出请求，输出了多个工具调用请求。也称为多结构化输出错误。

给出的提示词包含两个可提取结构化内容的文本，通过ToolStrategy的Union类型，模型可能会调用两次Function Call，但这是不允许的，此时报错，观察报错信息。


- 设置为True，报错信息返回给模型

- 设置为False，中断程序

- 设置为固定字符串，返回固定字符串给模型

In [2]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model(
    model = "gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
)

model.invoke("你好")

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=True
        #handle_errors="请检查输入数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang@atguigu.com，活动名称：公司年会，活动日期：2
026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='9d115ee8-7a83-4150-9a13-e9c8db1b3a60'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 70,
                    'prompt_tokens': 125,
                    'total_tokens': 195,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00040875,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00040875,
                        'upstream_inference_prompt_cost': 9.375e-05,
                        'upstream_inference_completions_cost': 0.000315
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784347548-wOZBhbymvOjUXJQQ4YNA',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f7366-cf74-7200-909b-c1fcf813c5a3-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang@atguigu.com'},
                    'id': 'call_VoILWuLegQcpEy5OrJtwPzOh',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_OSz1W4zcMGQdaVQfw7t0HR6u',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 125,
                'output_tokens': 70,
                'total_tokens': 195,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='a67c85bc-fb2a-4d71-9c9b-cedc8f263535',
            tool_call_id='call_VoILWuLegQcpEy5OrJtwPzOh'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='04f429dc-8e7c-4264-bab9-863128475c8c',
            tool_call_id='call_OSz1W4zcMGQdaVQfw7t0HR6u'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 28,
                    'prompt_tokens': 267,
                    'total_tokens': 295,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction

####  情况2：设置为指定异常类型

默认情况下，LangChain会处理结构化输出处理时抛出的两类异常：

1. MultipleStructuredOutputsError
多结构化输出错误，当返回的工具调用请求数量大于1时，抛出该异常，默认情况下LangChain会拦截异常并提醒模型重试。

2. StructuredOutputValidationError
输出结构化验证错误，当输出格式不符合结构化要求时，抛出上述异常。默认情况下 LangChain 会拦截该异常并自动重试。

In [ ]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy,
    StructuredOutputValidationError, MultipleStructuredOutputsError
from rich import print as rprint

class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")

class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=
        (MultipleStructuredOutputsError,StructuredOutputValidationError)
        # handle_errors=(StructuredOutputValidationError)
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

#### 情况3：设置为自定义错误处理函数

指定异常处理函数并返回字符串时，LangChain 会在遇到异常时自动重试并将异常处理函数的返回值作为 ToolMessage 的内容。
我们也可以选择在异常处理函数中抛出异常。

In [3]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy,StructuredOutputValidationError, MultipleStructuredOutputsError
from rich import print as rprint

# 自定义错误处理函数
def custom_error_handler(error: Exception) -> str:
    """自定义错误处理器"""
    error_str = str(error)

    print(f"捕获到错误类型: {type(error).__name__}")
    print(f"错误详情: {error_str}")

    if isinstance(error, StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符合要求。"
    elif isinstance(error, MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个进行返回。"
    else:
        return f"Error: {error_str}"


class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=custom_error_handler
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": f"请提取以下文本中内容：姓名：张三，电子邮箱：zhang@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"
    }]
})

rprint(result)

捕获到错误类型: MultipleStructuredOutputsError
错误详情: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.


{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang@atguigu.com，活动名称：公司年会，活动日期：2
026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='9d11c741-2158-4e8a-8905-792f62b8f92c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 70,
                    'prompt_tokens': 125,
                    'total_tokens': 195,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00040875,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00040875,
                        'upstream_inference_prompt_cost': 9.375e-05,
                        'upstream_inference_completions_cost': 0.000315
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1784348896-voKH1dFnFmizkSkDs25W',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f737b-5dba-78d0-a92a-3d1a8beb6685-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang@atguigu.com'},
                    'id': 'call_nIvRL7mQaZf07akfBbZjdX4r',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_3uQCG1GabY7cDh1AszxBYueH',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 125,
                'output_tokens': 70,
                'total_tokens': 195,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='检测到多个响应，请选择最相关的一个进行返回。',
            name='ContactInfo',
            id='b4b6f7fe-ad9d-4d96-ab4c-a8b41c6bdb82',
            tool_call_id='call_nIvRL7mQaZf07akfBbZjdX4r'
        ),
        ToolMessage(
            content='检测到多个响应，请选择最相关的一个进行返回。',
            name='EventDetails',
            id='092db856-53e2-42f7-b57d-dfa74bac3382',
            tool_call_id='call_3uQCG1GabY7cDh1AszxBYueH'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 28,
                    'prompt_tokens': 241,
                    'total_tokens': 269,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_writ